# 08 — Bonus Features
**GeoMetric Project**

Covers all bonus implementations:
- Spatial Autocorrelation (Moran's I + LISA clusters)
- Animated Temporal Choropleth (1990 → 2020)
- Interactive Plotly Dashboard
- Network Centrality deep-dive

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import warnings; warnings.filterwarnings("ignore")
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from scripts.utils.config import PATHS, STYLE
from scripts.utils.map_utils import reproject_gdf, save_figure, add_map_annotations

In [ ]:
# ── BONUS 1: Moran's I ──────────────────────────────────────
world = gpd.read_file(PATHS["processed"] / "master_world.gpkg")
world = world.dropna(subset=["co2_per_capita"]).copy()
world["geometry"] = world["geometry"].buffer(0)
if world.crs.to_epsg() != 4326:
    world = world.to_crs("EPSG:4326")

try:
    from libpysal.weights import Queen
    from esda.moran import Moran, Moran_Local

    w = Queen.from_dataframe(world, silence_warnings=True)
    w.transform = "R"
    y = world["co2_per_capita"].values
    mi = Moran(y, w)

    print(f"=== Global Moran's I ===")
    print(f"  I statistic : {mi.I:.4f}  (range: -1 to +1)")
    print(f"  Expected I  : {mi.EI:.4f}")
    print(f"  z-score     : {mi.z_norm:.4f}")
    print(f"  p-value     : {mi.p_norm:.6f}")
    print(f"  Significance: {'✅ SIGNIFICANT (p<0.05)' if mi.p_norm < 0.05 else '❌ Not significant'}")
    print(f"\n  Interpretation: CO₂ emissions cluster spatially")
    print(f"  High-emission countries are near other high-emission countries")

except ImportError:
    print("Install with: pip install libpysal esda")

In [ ]:
# LISA cluster map
try:
    import matplotlib.patches as mpatches
    lm = Moran_Local(y, w, seed=42)
    sig = lm.p_sim < 0.05

    world["lisa"] = "Not significant"
    world.loc[sig & (lm.q==1), "lisa"] = "High-High"
    world.loc[sig & (lm.q==2), "lisa"] = "Low-High"
    world.loc[sig & (lm.q==3), "lisa"] = "Low-Low"
    world.loc[sig & (lm.q==4), "lisa"] = "High-Low"

    LCOLORS = {"High-High":"#d73027","Low-Low":"#4575b4",
               "Low-High":"#abd9e9","High-Low":"#fdae61","Not significant":"#eeeeee"}
    world["lcolor"] = world["lisa"].map(LCOLORS)

    world_proj = reproject_gdf(world, "robinson")
    fig, ax = plt.subplots(figsize=(18,9))
    fig.patch.set_facecolor("white")
    ax.set_facecolor(STYLE["ocean_color"])
    for color, grp in world_proj.groupby("lcolor"):
        grp.plot(ax=ax, color=color, linewidth=0.3, edgecolor="#888")

    patches = [mpatches.Patch(color=c, label=l) for l,c in LCOLORS.items()]
    ax.legend(handles=patches, title="LISA Cluster (p<0.05)", loc="lower left", fontsize=9)
    ax.set_axis_off()
    add_map_annotations(ax, title=f"Local Moran's I — CO₂ per Capita Clusters (Global I={mi.I:.3f}, p={mi.p_norm:.4f})",
                        subtitle="High-High = emission hotspot cluster | Low-Low = low-emission cluster",
                        source="OWID 2020 / PySAL", projection_name="robinson", year=2020)
    save_figure(fig, PATHS["fig_bonus"] / "map_lisa_clusters.png")
    plt.show()
except Exception as e:
    print(f"Skipped: {e}")

In [ ]:
# ── BONUS 2: Animated choropleth ───────────────────────────
emiss = pd.read_csv(PATHS["raw_emissions"] / "owid-co2-data.csv", low_memory=False)
years = list(range(1990, 2021, 5))
df = emiss[
    emiss["year"].isin(years) &
    emiss["iso_code"].notna() &
    ~emiss["iso_code"].str.startswith("OWID")
][["iso_code","country","year","co2_per_capita","population"]].dropna(subset=["co2_per_capita"])
df["year"] = df["year"].astype(str)

fig = px.choropleth(
    df, locations="iso_code", color="co2_per_capita",
    color_continuous_scale="YlOrRd", range_color=[0,20],
    animation_frame="year", hover_name="country",
    hover_data={"co2_per_capita":":.2f","population":":,.0f","iso_code":False},
    title="GeoMetric — CO₂ per Capita 1990→2020 (Animated)",
    labels={"co2_per_capita":"CO₂/capita (t)"}, projection="natural earth",
)
fig.update_geos(showcoastlines=True, coastlinecolor="white",
                showland=True, landcolor="#f5f5f0",
                showocean=True, oceancolor="#cfe2f3")
fig.update_layout(height=550, margin={"r":0,"t":50,"l":0,"b":0})

out = PATHS["interactive_anim"] / "co2_animation.html"
fig.write_html(str(out), include_plotlyjs="cdn")
print(f"✅ Animation saved: {out}")
fig.show()

In [ ]:
# ── BONUS 3: Interactive Plotly map ────────────────────────
world_wgs = gpd.read_file(PATHS["processed"] / "master_world.gpkg")
if world_wgs.crs.to_epsg() != 4326:
    world_wgs = world_wgs.to_crs("EPSG:4326")

wj = world_wgs.__geo_interface__
df_plot = world_wgs[["iso_a3","country_name","co2_per_capita","pop_final","gdp_per_capita"]].dropna(subset=["co2_per_capita"])

fig = px.choropleth(
    df_plot, geojson=wj, locations="iso_a3",
    featureidkey="properties.iso_a3",
    color="co2_per_capita", color_continuous_scale="YlOrRd",
    hover_name="country_name",
    hover_data={"iso_a3":False,"co2_per_capita":":.2f",
                "pop_final":":,.0f","gdp_per_capita":":.0f"},
    title="GeoMetric — Interactive CO₂ per Capita Explorer",
    labels={"co2_per_capita":"CO₂/capita (t)","pop_final":"Population","gdp_per_capita":"GDP/capita"},
    projection="natural earth",
)
fig.update_geos(showcoastlines=True, coastlinecolor="white",
                showland=True, landcolor="#f0ede4",
                showocean=True, oceancolor="#cfe2f3")
fig.update_layout(height=550, margin={"r":0,"t":50,"l":0,"b":0})

out = PATHS["interactive_folium"] / "co2_interactive_choropleth.html"
fig.write_html(str(out), include_plotlyjs="cdn")
print(f"✅ Saved: {out}")
fig.show()